# A tiny variational classifier
This one-qubit example has no quantum advantage: two same-axis rotations combine into one. We use exact Qiskit statevectors to isolate the training calculation, then separately examine shot uncertainty.

Goal: encode x, predict P(1), differentiate by parameter shift, train on one subset, and evaluate once on a held-out subset. The threshold baseline is appropriate for this deliberately simple generated task.

In [ ]:
%pip -q install "qiskit>=2,<3" "numpy>=1.26,<3" "matplotlib>=3.8,<4"


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector
rng = np.random.default_rng(42)
def prediction(x, w):
    qc = QuantumCircuit(1)
    qc.ry(float(x), 0); qc.ry(float(w), 0)
    return float(Statevector.from_instruction(qc).probabilities()[1])
def gradient(x, w):
    return (prediction(x,w+np.pi/2)-prediction(x,w-np.pi/2))/2
for x in [-1.,0.,1.]:
    assert np.isclose(prediction(x,.3), np.sin((x+.3)/2)**2)
    eps=1e-5
    assert np.isclose(gradient(x,.3), (prediction(x,.3+eps)-prediction(x,.3-eps))/(2*eps), atol=1e-8)

In [ ]:
X = rng.uniform(-1.3, 1.3, 80)
y = (X > 0).astype(float)
indices = rng.permutation(len(X))
train, test = indices[:60], indices[60:]
w = 0.2
losses = []
for epoch in range(100):
    probs = np.array([prediction(x,w) for x in X[train]])
    grads = np.array([gradient(x,w) for x in X[train]])
    losses.append(np.mean((probs-y[train])**2))
    # MSE chain rule: 2(p-y) dp/dw.
    w -= 0.3 * np.mean(2*(probs-y[train])*grads)
assert losses[-1] < losses[0]
held_out = np.array([prediction(x,w) for x in X[test]])
accuracy = np.mean((held_out >= .5)==y[test])
baseline = np.mean((X[test] > 0)==y[test])
print(f"Trained w={w:.3f}; test accuracy={accuracy:.3f}")
print(f"Classical threshold baseline accuracy={baseline:.3f}")
plt.plot(losses); plt.xlabel("Training epoch"); plt.ylabel("Training MSE"); plt.show()

In [ ]:
p = prediction(.4,w)
for shots in [100,1000,10000]:
    estimate = rng.binomial(shots,p)/shots
    se = np.sqrt(p*(1-p)/shots)
    print(f"N={shots}: sampled={estimate:.4f}, ideal={p:.4f}, theoretical standard error={se:.4f}")

## Your turn
1. Train with a different seed. Report test accuracy without tuning on the test set.
2. Use labels based on |x| rather than x. What can this ansatz represent?
3. Plot finite-shot gradient estimates. How does the shot budget affect training?
4. Explain why shifting the MSE itself is not the parameter-shift calculation used here.

Do not interpret this toy result as evidence of practical quantum speedup.